In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
!pip install -q fastapi uvicorn pyngrok transformers==4.52.4 accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
NGROK_TOKEN = "3KStzwJ1hrzx2y57jMVHm0TI0lQ_zk6gSGYu3Z6LhVppga1t"
API_KEY = "secret123"

MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch

from fastapi import FastAPI, Request, HTTPException

import uvicorn
import threading
import time
import socket

from pyngrok import ngrok, conf

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

print("Model loaded successfully!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!


In [9]:
def generate_text(prompt, max_length=300):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [10]:
def generate_question(topic):

    prompt = f"""
You are Presentation Coach.

The student is preparing a presentation about:

{topic}

Generate ONE useful question that a teacher
or audience might ask after the presentation.

The question should be clear and suitable
for a university student.

Return ONLY the question.
"""

    return generate_text(
        prompt,
        max_length=100
    )

In [11]:
question = generate_question(
    "Artificial Intelligence"
)

print(question)


You are Presentation Coach.

The student is preparing a presentation about:

Artificial Intelligence

Generate ONE useful question that a teacher
or audience might ask after the presentation.

The question should be clear and suitable
for a university student.

Return ONLY the question.
What are the potential ethical implications of using AI in decision-making processes, particularly in fields like healthcare or criminal justice, and how can we ensure that these systems are fair and unbiased?


In [12]:
def evaluate_answer(topic, question, answer):

    prompt = f"""
You are Presentation Coach.

Evaluate the student's answer.

Topic:
{topic}

Question:
{question}

Student Answer:
{answer}

Give the evaluation using exactly this format:

SCORE: X/10

STRENGTHS:
- point 1
- point 2

WEAKNESSES:
- point 1
- point 2

MISSING:
- point 1

ADVICE:
one short useful advice

Keep the feedback simple and useful
for a university student.
"""

    return generate_text(
        prompt,
        max_length=300
    )

In [13]:
topic = "Artificial Intelligence"

question = "What is Artificial Intelligence?"

answer = """
Artificial Intelligence is a technology that allows
computers to perform tasks that normally require
human intelligence, such as learning and decision making.
"""

result = evaluate_answer(
    topic,
    question,
    answer
)

print(result)


You are Presentation Coach.

Evaluate the student's answer.

Topic:
Artificial Intelligence

Question:
What is Artificial Intelligence?

Student Answer:

Artificial Intelligence is a technology that allows
computers to perform tasks that normally require
human intelligence, such as learning and decision making.


Give the evaluation using exactly this format:

SCORE: X/10

STRENGTHS:
- point 1
- point 2

WEAKNESSES:
- point 1
- point 2

MISSING:
- point 1

ADVICE:
one short useful advice

Keep the feedback simple and useful
for a university student.
---

SCORE: 7/10

STRENGTHS:
- Provides a clear and concise definition of AI.
- Uses specific examples of tasks that AI can perform.

WEAKNESSES:
- Does not mention the historical context or evolution of AI.
- Does not discuss the different types of AI (e.g., narrow, general, super).

MISSING:
- Explanation of how AI learns and makes decisions.

ADVICE:
To improve your answer, consider discussing the different types of AI and providing a b

In [14]:
app = FastAPI()

In [15]:
@app.post("/question")
async def question_api(req: Request):

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    data = await req.json()

    topic = data.get("topic", "")

    question = generate_question(topic)

    return {
        "question": question
    }

In [16]:
@app.post("/evaluate")
async def evaluate_api(req: Request):

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    data = await req.json()

    topic = data.get("topic", "")
    question = data.get("question", "")
    answer = data.get("answer", "")

    result = evaluate_answer(
        topic,
        question,
        answer
    )

    return {
        "evaluation": result
    }

In [17]:
@app.get("/health")
async def health():

    return {
        "status": "ok",
        "project": "Presentation Coach"
    }

In [18]:
def free_port():

    s = socket.socket()

    s.bind(('', 0))

    port = s.getsockname()[1]

    s.close()

    return port


port = free_port()

print("Port:", port)

Port: 35317


In [19]:
conf.get_default().auth_token = NGROK_TOKEN

public_url = ngrok.connect(port).public_url

print("Public URL:", public_url)

Public URL: https://uprising-education-overcoat.ngrok-free.dev                                      


In [20]:
def run():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


threading.Thread(
    target=run,
    daemon=True
).start()

time.sleep(2)

print("Presentation Coach server is running!")

INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:35317 (Press CTRL+C to quit)


Presentation Coach server is running!


In [21]:
import requests

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "ngrok-skip-browser-warning": "true"
}

response = requests.get(
    public_url + "/health",
    headers=headers,
    timeout=30
)

print(response.json())

INFO:     35.237.58.120:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'project': 'Presentation Coach'}
INFO:     197.54.179.20:0 - "POST /question HTTP/1.1" 200 OK
INFO:     197.54.179.20:0 - "POST /question HTTP/1.1" 200 OK
INFO:     197.54.179.20:0 - "POST /question HTTP/1.1" 200 OK
